In [ ]:
#
# Claudio Rodriguez Meza
#
# *******************************
# lo comprendido en el ejercicio
# *******************************
# - Cargar desde 2025-01-01 a la fecha
# - Carga diaria de manera incremental
# - Cargar 4 securitys, con valores vigentes al dia  (cuidado con UDIMX.SPOT)
# - Crear tabla estadistica de los ultimos 7 dias calendario
# 	DUDA: ¿desde la ultima fecha del rango indicado? 
# 		  por ejemplo si el ultimo dia del rango es 2027, pero el ultimo registro de cada serie puede ser distinto. se 
# 
# - Almacenar la respuesta de la api como Raw Parquet
# - Corren N veces, sin corrupcion o duplicidad
# - Generar loggin



In [ ]:
#
# SE CREA UNA CLASE "miETL", que contiene todas las funciones necesarias para el ejericio
# 
# -----------------------------------------
# PRIMERO Ejecutar la celda de la clase
# -----------------------------------------

In [ ]:
#from vmetrix import get_database   
#**** NO UTILIZARE ESA CLASE porque las tablas deben estar preconstruidas, y dependen del DF. sobretodo para la tabla staging raw
#
from vmetrix.config import get_config
from vmetrix import get_banxico_api
import logging
import duckdb
import json
from datetime import datetime
import pandas as pd

archlog=datetime.now().strftime("%Y%m%d")

logging.basicConfig(
    filename = f"S3/logs/pipeline_{archlog}.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
# DEBUG < INFO < WARNING < ERROR < CRITICAL

logger = logging.getLogger(__name__)


def fnc_validaFecha(fecha: str) -> bool:
    try:
        datetime.strptime(fecha, "%Y-%m-%d")
        return True
    except ValueError:
        return False
        
class miETL:
    # ----------------------------------------------------------
    # ----------------------------------------------------------
    def leeAPI(self, prmSeries: str, prmDesde: str, prmHasta: str):
        logger.info("class miETL ---Inicio leeAPI---")
    
        if not fnc_validaFecha(prmDesde) or not fnc_validaFecha(prmHasta):
            logger.error("class miETL ... Formato fechas no válido")
            return False, None
    
        api = get_banxico_api()
        sw_traeDatos = False;
        
        try:
            resultadoJSON = api.get_values_between(
                series=prmSeries,
                start_date=prmDesde,
                end_date=prmHasta
            )
        except Exception as e:
            logger.exception("class miETL ...leeAPI, Error llamando API: %s", e)
            return False, None
    
        if not isinstance(resultadoJSON, dict):
            logger.error("class miETL ...leeAPI, Respuesta no es dict/json")
            return False, resultadoJSON
    
        bmx = resultadoJSON.get("bmx")
        if not isinstance(bmx, dict):
            logger.error("class miETL ...leeAPI, No existe clave 'bmx' o no es dict")
            return False, resultadoJSON
    
        series = bmx.get("series")
        if not isinstance(series, list):
            logger.error("class miETL ...leeAPI, No existe clave 'series' o no es lista")
            return False, resultadoJSON
    
        if len(series) == 0:
            logger.error("class miETL ...leeAPI, Lista 'series' vacía")
            return False, resultadoJSON
    
        for serie in series:
            id_serie = serie.get("idSerie", "SIN_ID")
    
            if "datos" not in serie:
                logger.warning("class miETL ...leeAPI, Serie %s no trae clave 'datos'", id_serie)
                #return False, resultadoJSON
                continue

            sw_traeDatos = True
            if not isinstance(serie["datos"], list):
                logger.warning("class miETL ...leeAPI, Serie %s trae 'datos' no lista", id_serie)
                #return False, resultadoJSON
    
            if len(serie["datos"]) == 0:
                logger.warning("class miETL ...leeAPI, Serie %s viene sin registros", id_serie)

        if sw_traeDatos == False:
            logger.error("class miETL ...leeAPI, el json no tiene ningun datos en las series")
            return False, resultadoJSON
            
        logger.info("class miETL ---Fin leeAPI OK---")
        return True, resultadoJSON    
    # ----------------------------------------------------------
    # ----------------------------------------------------------
    def haceParquetDesdeJson(self, prmJSON: dict, prmPath: str, prmFuente: str):
        logger.info("class miETL ... Inicio haceParquetDesdeJson")
        df_raw = pd.DataFrame([{
            "fuente": prmFuente,
            "raw_json": json.dumps(prmJSON, ensure_ascii=False)
        }])

        df_raw.to_parquet(prmPath, index=False)
        logger.info("class miETL ... Fin haceParquetDesdeJson generado en %s",prmPath)
        return True
        
    # ----------------------------------------------------------
    # ----------------------------------------------------------
    def haceParquetDuckDB(self, prmPath:str):
        # exporta de duck a parquet
        con = duckdb.connect("vmetrix/db.duckdb")

        instrsql=f"COPY (select * from tmp_raw_banxico where sw_reglaneg=0) TO '{prmPath}' (FORMAT PARQUET)"

        con.sql(instrsql)
        return True
        
    # ----------------------------------------------------------
    # ----------------------------------------------------------
    def cargaRawStageDB(self, prmJSON: dict, prmSwReglaNeg: bool, fechaFin: str):
        logger.info("class miETL ... cargaRawStageDB")

        rows = []
        for serie in prmJSON["bmx"]["series"]:
            id_serie = serie["idSerie"]
    
            if "datos" not in serie:
                logger.warning("class miETL ... Serie %s no trae 'datos'",id_serie)
                continue
        
            for item in serie["datos"]:
                rows.append({
                    "idSerie": id_serie,
                    "fecha": item["fecha"],
                    "dato": item["dato"]
                })

        df_raw = pd.DataFrame(rows)

        logger.info("class miETL ... se insertan %i registros en Db raw", len(df_raw))
        

        con = duckdb.connect("vmetrix/db.duckdb")
        con.register("df_raw_view", df_raw)
        con.sql("""
        drop table if exists tmp_raw_banxico;
        CREATE TABLE IF NOT EXISTS tmp_raw_banxico (
            idSerie VARCHAR,
            fecha VARCHAR,
            dato VARCHAR,
            sw_reglaNeg int,
            obs_reglaNeg varchar
        )
        """)
        con.sql("""
        INSERT INTO tmp_raw_banxico SELECT idSerie, fecha, dato, 0, '' FROM df_raw_view
        """)

        if prmSwReglaNeg==True:
            logger.info("class miETL ... se realiza transformacion y reglas de negocio")

            con.sql("""
            update tmp_raw_banxico set sw_reglaNeg=1, obs_reglaNeg='Fecha no valida en campo fecha'
            where TRY_STRPTIME(fecha, '%d/%m/%Y') IS NULL
            """)
            
            con.sql("""
            update tmp_raw_banxico set sw_reglaNeg=1, obs_reglaNeg='Decimal no valido en campo dato'
            where TRY_CAST(dato AS DECIMAL(18,6)) is null
            """)
            
            con.sql("""
            update tmp_raw_banxico set sw_reglaNeg=2, obs_reglaNeg='Fecha excede el limite'
            where TRY_STRPTIME(fecha, '%d/%m/%Y')>CAST(? AS DATE)
            """, params=[fechaFin])
            
        return True

    # ----------------------------------------------------------
    def cargaHistDB(self):
        logger.info("class miETL ... Inicio cargaHistDB")
        con = duckdb.connect("vmetrix/db.duckdb")

        con.sql("""
        CREATE TABLE IF NOT EXISTS tbl_hist_banxico (
            idSerie VARCHAR,
            fecha DATE,
            dato decimal(18,6)
        )
        """)
        
        con.sql("""
        DELETE FROM tbl_hist_banxico AS his
        USING tmp_raw_banxico AS tmp
        WHERE 
            tmp.sw_reglaneg = 0
            AND tmp.idSerie = his.idSerie
            AND STRPTIME(tmp.fecha, '%d/%m/%Y') = his.fecha;
        """)
        
        con.sql("""
        INSERT INTO tbl_hist_banxico
        SELECT
            idSerie,
            STRPTIME(fecha, '%d/%m/%Y')::DATE AS fecha,
            CAST(dato AS decimal(18,6)) AS dato
        FROM tmp_raw_banxico
        where sw_reglaNeg=0
        """)
        logger.info("class miETL ... FIN cargaHistDB")

        return True


In [1]:
#--- para eliminar las tablas utilizadas en duckdb (OPCIONAL)
#---------------------------------------------------------------
#import duckdb
#con = duckdb.connect("vmetrix/db.duckdb")
#con.sql("""drop table if exists tbl_hist_banxico""")
#con.sql("""drop table if exists tmp_raw_banxico""")
#con.sql("""show tables""")





┌────────────────┐
│      name      │
│    varchar     │
├────────────────┤
│ BANXICO_SERIES │
└────────────────┘

In [ ]:
#
# ----------------------------------------------------------
# SEGUNDO , ejecutar cambiando solo 4 parametros iniciales
# ----------------------------------------------------------
#


# Parametros de Ejecucion
# -----------------------
seriesExtraer = "SF43718,SP68257,SF331451,SF43773"
fechaIni = "2025-01-01" 
fechaFin = "2026-01-01"
ejecutaReglasNegocio = True

#
# vars
fecha_hora_exec=datetime.now().strftime("%Y%m%d_%H%M%S")
arch1_Parquet = f"S3/parquets/raw_orginal_api_banxico_{fecha_hora_exec}.parquet"
arch2_Parquet = f"S3/parquets/duckdb_ingesta_api_banxico_{fecha_hora_exec}.parquet"


######## 
# Paso 1, lee la api, y si el resultado es positivo (existe json con datos), guarda el parquet
########
exeETL = miETL()
swReturn, resultadoJSON = exeETL.leeAPI(seriesExtraer, fechaIni, fechaFin)
if swReturn==True:
    #arch1_Parquet
    exeETL = miETL()
    swReturn = exeETL.haceParquetDesdeJson(resultadoJSON, arch1_Parquet, "banxico_api")    

######## 
# Paso 2, Carga el json en una base duckdb de staging  (tabla tmp_raw_banxico) 
# SE PUEDE EVITAR LA VALIDACION REGLA DE NEGOCIO (False)
########
if swReturn==True:
    swReturn = exeETL.cargaRawStageDB(resultadoJSON, True, fechaFin)


######## 
# Paso 3, Carga finalmente en base duckdb se guarda una base HISTORICA (tabla tbl_hist_banxico) sin duplicar y guarda el parquet procesado
########
if swReturn==True:
    swReturn = exeETL.cargaHistDB()
if swReturn==True:
    swReturn = exeETL.haceParquetDuckDB(arch2_Parquet)

######## 
# Paso 4, ESTADISTICAS
########
if swReturn==True:
    con = duckdb.connect("vmetrix/db.duckdb")
    
    print("")
    print("****** Resumen de carga historica por, serie y año (tabla tbl_hist_banxico) ***************")
    con.sql("""
    Select year(fecha) as anio, idSerie, count(*) as cant_regs,min(fecha) as min_fec,max(fecha) as max_fec
    from tbl_hist_banxico
    group by idSerie,year(fecha)
    order by 1,2
    """).show()

    print("****** lo cargado en staging (tabla tmp_raw_banxico) ***************")
    con.sql("""
        Select idSerie
            , min(STRPTIME(tmp.fecha, '%d/%m/%Y')) as desde
            , max(STRPTIME(tmp.fecha, '%d/%m/%Y')) as hasta
            , sum(case when sw_reglaneg=0 then 1 else 0 end) regs_procesar
            , sum(case when sw_reglaneg=0 then 0 else 1 end) regs_regla_neg
            , count(*) total_regs
        from tmp_raw_banxico tmp
        group by idSerie
        order by 1
        """).show()

    print("****** Muestra de los ultimos 7 dias de cada serie, desde la ultima fecha en tabla historica - 7 dias (tabla tbl_hist_banxico) ******")
    
    con.sql("""
    Select his.idSerie, his.fecha, his.dato
    from tbl_hist_banxico his
    inner join (
        Select idSerie, max(fecha) as max_fec, max(fecha) - interval 7 days as resta_dias
        from tbl_hist_banxico
        group by idSerie
        ) res on res.idSerie=his.idSerie and his.fecha>=res.resta_dias
    order by 1,2    
    """).show()

In [ ]:
#------------------------
# leer la tabla exportada con parquet
#------------------------
#fecha_hora_exec=datetime.now().strftime("%Y%m%d_%H%M%S")
#arch2_Parquet = f"S3/parquets/duckdb_ingesta_api_banxico_{fecha_hora_exec}.parquet"

#con.sql(f"""
#SELECT * 
#FROM read_parquet('{arch2_Parquet}')
#""").show()

#--------------------------------
# prueba leer json orig con parquet
#--------------------------------

#arch = "S3/parquets/raw_orginal_api_banxico_20260425_192138.parquet"
#con.sql(f"""
#SELECT * 
#FROM read_parquet('{arch}')
#""").show()